In [4]:
import io
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import geopandas as gpd
import requests

# Full USGS gauge inventory (all states) used elsewhere in this project — has
# per-site period-of-record (begin/end/n_years) already computed.
DATA_DIR = Path("..") / "Hawaii results"
GAUGE_PARQUET = DATA_DIR / "usgs_lt500_min2014-2024.parquet"

FETCH_TIMEOUT = 20   # seconds per NWIS request
FETCH_WORKERS = 10   # parallel HTTP connections

## 1. Period of record — how far back does each Hawaii gauge go?

Loads the pre-built USGS gauge inventory and filters it to Hawaii (`state_abbr == "HI"`).
This inventory already has `begin` / `end` / `n_years` computed per site, so no NWIS
request is needed just to answer this question.

In [5]:
gdf = gpd.read_parquet(GAUGE_PARQUET)
hi = gdf[gdf["state_abbr"] == "HI"].copy()
hi["site_no"] = hi["id"].str.replace("usgs-", "", regex=False)
hi["begin"] = pd.to_datetime(hi["begin"])
hi["end"] = pd.to_datetime(hi["end"])

record_length = (
    hi[["site_no", "name", "begin", "end", "n_years", "area_km2"]]
    .sort_values("begin")
    .reset_index(drop=True)
)

print(f"Found {len(record_length)} USGS gauges in Hawaii")
print(f"Earliest record starts: {record_length['begin'].min().date()}")
print(f"Longest period of record: {record_length['n_years'].max()} years "
      f"({record_length.loc[record_length['n_years'].idxmax(), 'name']})")

record_length

Found 58 USGS gauges in Hawaii
Earliest record starts: 1981-09-29
Longest period of record: 45 years (Kaluanui Stream nr Punaluu, Oahu, HI)


,site_no,name,begin,end,n_years,area_km2
0,16304200,"Kaluanui Stream nr Punaluu, Oahu, HI",1981-09-29 10:00:00,2026-03-10 11:15:00,45,2.8
1,16226200,"N. Halawa Str nr Honolulu, Oahu, HI",1986-07-19 10:00:00,2026-03-10 07:15:00,40,10.4
2,16213000,"Waikele Str at Waipahu, Oahu, HI",1986-10-01 10:30:00,2026-03-10 07:00:00,40,116.9
3,16345000,"Opaeula Str nr Wahiawa, Oahu, HI",1986-11-23 10:30:00,2026-03-10 04:55:00,40,7.8
4,16272200,"Kamooalii Str blw Luluku Str nr Kaneohe, Oahu, HI",1990-10-01 10:15:00,2025-01-30 10:00:00,35,9.9
5,16725000,"Alakahi Stream near Kamuela, HI",1990-10-01 10:25:15,2026-03-10 10:10:00,36,2.1
6,16031000,"Waimea River near Waimea, Kauai, HI",1990-10-01 10:30:00,2026-03-10 10:30:00,36,149.3
7,16019000,"Waialae Str at alt 3,820 ft nr Waimea, Kauai, HI",1990-10-01 10:30:00,2026-03-10 10:55:00,36,5.4
8,16620000,"Honokohau Stream near Honokohau, Maui, HI",1990-10-01 10:30:00,2026-03-10 10:45:00,36,10.8
9,16108000,"Wainiha River nr Hanalei, Kauai, HI",1990-10-01 10:30:00,2026-03-10 10:50:00,36,27.0


## 2. Fetch annual peak-streamflow records for every Hawaii gauge

USGS publishes one instantaneous peak discharge per water-year per site via the
NWIS **peak-flow** service — this is the standard source for "biggest flood on
record" style questions, and is far cheaper to pull than full instantaneous/daily
records for 60+ years × 30+ sites.

Uses the same parallel-fetch + hand-rolled RDB parsing pattern already used for
rating curves in `utils.py`.

In [6]:
NWIS_PEAK_URL = "https://nwis.waterdata.usgs.gov/nwis/peak?site_no={site_no}&agency_cd=USGS&format=rdb"


def fetch_peak_flows(site_no: str):
    """
    Fetch the annual peak-streamflow record (RDB format) for one USGS gauge.

    Returns a DataFrame with columns [site_no, peak_dt, peak_va, gage_ht] — one
    row per water-year peak — or None if the request fails / no data exists.
    """
    url = NWIS_PEAK_URL.format(site_no=site_no)
    try:
        resp = requests.get(url, timeout=FETCH_TIMEOUT)
        resp.raise_for_status()
    except Exception as exc:
        print(f"  [warn] peak-flow fetch failed ({site_no}): {exc}")
        return None

    # RDB format: comment lines start with '#', then a header row, then a
    # column-width spec row (e.g. "5s 15s 10d ..."), then the data rows.
    lines = [ln for ln in resp.text.splitlines() if ln and not ln.startswith("#")]
    if len(lines) < 3:
        return None   # header + spec row only — no peak data for this site

    try:
        df = pd.read_csv(io.StringIO("\n".join(lines)), sep="\t", dtype=str)
    except Exception as exc:
        print(f"  [warn] could not parse peak-flow RDB ({site_no}): {exc}")
        return None

    df = df[df["site_no"].astype(str).str.match(r"^\d+$")]   # drop RDB spec row
    if df.empty:
        return None

    df["peak_dt"] = pd.to_datetime(df["peak_dt"], errors="coerce")
    df["peak_va"] = pd.to_numeric(df["peak_va"], errors="coerce")
    df = df.dropna(subset=["peak_dt", "peak_va"])
    return df[["site_no", "peak_dt", "peak_va", "gage_ht"]] if not df.empty else None


def fetch_all_peak_flows(site_nos):
    """Fetch peak-flow records for all sites in parallel; returns one concatenated DataFrame."""
    print(f"Fetching peak-flow records for {len(site_nos)} gauges…")
    results = []
    with ThreadPoolExecutor(max_workers=FETCH_WORKERS) as pool:
        futures = {pool.submit(fetch_peak_flows, s): s for s in site_nos}
        for fut in as_completed(futures):
            df = fut.result()
            if df is not None and not df.empty:
                results.append(df)

    combined = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
    n_sites = combined["site_no"].nunique() if not combined.empty else 0
    print(f"  Retrieved {len(combined)} peak-flow records from {n_sites}/{len(site_nos)} gauges")
    return combined


peak_flows = fetch_all_peak_flows(hi["site_no"].tolist())
peak_flows.head()

Fetching peak-flow records for 58 gauges…
  Retrieved 3506 peak-flow records from 58/58 gauges


,site_no,peak_dt,peak_va,gage_ht
0,16212490,2013-10-14,611.0,8.64
1,16212490,2015-08-26,418.0,8.07
2,16212490,2016-08-24,39.0,5.93
3,16212490,2017-02-11,49.9,6.06
4,16212490,2017-12-04,44.7,6.00


## 3. Top 20 largest streamflow events across all Hawaii gauges

Combines the peak-flow records from every gauge into one table and ranks all
events by peak discharge (cfs), regardless of which station recorded them.

In [7]:
peak_flows_named = peak_flows.merge(
    hi[["site_no", "name", "area_km2"]], on="site_no", how="left"
)

top20 = (
    peak_flows_named.sort_values("peak_va", ascending=False)
    .head(20)[["site_no", "name", "peak_dt", "peak_va", "gage_ht", "area_km2"]]
    .rename(columns={"peak_va": "peak_discharge_cfs", "gage_ht": "gage_height_ft"})
    .reset_index(drop=True)
)
top20.index += 1   # rank 1 = largest event

print("Top 20 largest recorded streamflow events — all Hawaii gauges combined")
top20

Top 20 largest recorded streamflow events — all Hawaii gauges combined


,site_no,name,peak_dt,peak_discharge_cfs,gage_height_ft,area_km2
1,16060000,"SF Wailua River nr Lihue, Kauai, HI",1963-04-15,68800.0,23.10,61.5
2,16060000,"SF Wailua River nr Lihue, Kauai, HI",1955-11-12,46700.0,11.52,61.5
3,16060000,"SF Wailua River nr Lihue, Kauai, HI",1920-01-16,44900.0,11.25,61.5
4,16103000,"Hanalei River nr Hanalei, Kauai, HI",1995-11-03,41600.0,15.81,47.9
5,16060000,"SF Wailua River nr Lihue, Kauai, HI",2012-03-04,39200.0,20.54,61.5
6,16060000,"SF Wailua River nr Lihue, Kauai, HI",1990-11-18,38600.0,20.47,61.5
7,16031000,"Waimea River near Waimea, Kauai, HI",1949-02-07,37100.0,19.30,149.3
8,16060000,"SF Wailua River nr Lihue, Kauai, HI",1921-01-16,36400.0,10.40,61.5
9,16060000,"SF Wailua River nr Lihue, Kauai, HI",1975-01-30,36200.0,20.21,61.5
10,16060000,"SF Wailua River nr Lihue, Kauai, HI",1927-12-24,35300.0,10.30,61.5


## 4. Save results

Persist both tables so they can be reused without re-hitting NWIS.

In [8]:
record_length.to_csv(DATA_DIR / "hawaii_gauge_period_of_record.csv", index=False)
peak_flows_named.to_csv(DATA_DIR / "hawaii_all_peak_flows.csv", index=False)
top20.to_csv(DATA_DIR / "hawaii_top20_peak_events.csv")

print(f"Saved period-of-record table -> {DATA_DIR / 'hawaii_gauge_period_of_record.csv'}")
print(f"Saved all peak-flow records  -> {DATA_DIR / 'hawaii_all_peak_flows.csv'}")
print(f"Saved top-20 peak events     -> {DATA_DIR / 'hawaii_top20_peak_events.csv'}")

Saved period-of-record table -> ..\Hawaii results\hawaii_gauge_period_of_record.csv
Saved all peak-flow records  -> ..\Hawaii results\hawaii_all_peak_flows.csv
Saved top-20 peak events     -> ..\Hawaii results\hawaii_top20_peak_events.csv
